In [1]:
import requests
from typing import List, Dict
import pandas as pd
from sqlalchemy import create_engine,MetaData, Table, Column, Integer, String, ForeignKey , Numeric, Date , TEXT, Boolean
from dotenv import load_dotenv
import os

In [2]:
BASE_URL = "http://localhost:8000"
load_dotenv()
WAREHOUSE_URL = os.getenv("WAREHOUSE_URL")

In [ ]:
def fetch_all_pages(endpoint: str, page_size: int = 10000) -> List[Dict]:
    all_data = []
    page = 1

    while True:
        response = requests.get(
            f"{BASE_URL}{endpoint}",
            params={"page": page, "page_size": page_size},
            timeout=10
        )
        
        if response.status_code != 200:
            print(f"Error {response.status_code}: {response.text}")
            break

        result = response.json()
        items = result.get("data", [])
        
        if not items:
            break

        all_data.extend(items)
        print(f"Fetched page {page} → {len(items)} records")

        if len(items) < page_size:
            break

        page += 1

    df = pd.DataFrame(all_data)
    print(f"Total records fetched: {len(all_data)}")
    return df

In [5]:
DATE_COLUMNS = [
    "enroll_date",
    "expected_graduation",
    "session_date",
    "recorded_at",
    "enrolled_at",
    "start_date",
    "end_date"
]

def parse_date_columns(df):
    df = df.copy()

    for col in DATE_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    return df

In [ ]:
df_dim_students = fetch_all_pages("/export_students")
df_dim_courses = fetch_all_pages("/export_courses")
df_dim_enrollments = fetch_all_pages("/export_enrollments")
df_fact_attendance = fetch_all_pages("/export_attendance")
df_dim_course_offerings = fetch_all_pages("/export_course_offerings")
df_dim_department = fetch_all_pages("/export_department")

In [ ]:
def create_dim_date(start="2020-01-01", end="2035-12-31"):
    dates = pd.date_range(start=start, end=end)

    df = pd.DataFrame({
        "full_date": dates
    })

    df["date_key"] = df["full_date"].dt.strftime("%Y%m%d").astype(int)
    df["day"] = df["full_date"].dt.day
    df["month"] = df["full_date"].dt.month
    df["month_name"] = df["full_date"].dt.strftime("%B")
    df["year"] = df["full_date"].dt.year
    df["day_of_week"] = df["full_date"].dt.weekday + 1
    df["day_name"] = df["full_date"].dt.strftime("%A")
    df["week_of_year"] = df["full_date"].dt.isocalendar().week
    df["quarter"] = df["full_date"].dt.quarter
    df["is_weekend"] = df["day_of_week"].isin([6, 7])

    return df
df_dim_date = create_dim_date()

In [8]:
def convert_datetime_to_date_keys(df):
    df = df.copy()

    datetime_cols = df.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns

    for col in datetime_cols:
        df[f"{col}_date_key"] = df[col].dt.strftime("%Y%m%d").astype("Int64")

    return df

In [ ]:
df_dim_students = parse_date_columns(df_dim_students)
df_fact_attendance = parse_date_columns(df_fact_attendance)
df_dim_enrollments = parse_date_columns(df_dim_enrollments)
df_dim_course_offerings = parse_date_columns(df_dim_course_offerings)

In [ ]:
df_dim_students = convert_datetime_to_date_keys(df_dim_students)
df_fact_attendance = convert_datetime_to_date_keys(df_fact_attendance)
df_dim_enrollments = convert_datetime_to_date_keys(df_dim_enrollments)
df_dim_course_offerings = convert_datetime_to_date_keys(df_dim_course_offerings)

In [ ]:
df_fact_student = df_dim_students[["user_id","current_gpa","passed_credits","enroll_date_date_key"]]
df_fact_enrollment = df_dim_enrollments[["student_user_id","course_offering_id","enrolled_at_date_key"]]

In [ ]:
engine = create_engine(WAREHOUSE_URL)

In [ ]:
metadata = MetaData()

dim_date = Table(
    "dim_date", metadata,
    Column("date_key", Integer, primary_key=True),
    Column("full_date", Date),
    Column("day", Integer),
    Column("month", Integer),
    Column("month_name", String(20)),
    Column("year", Integer),
    Column("day_of_week", Integer),
    Column("day_name", String(20)),
    Column("week_of_year", Integer),
    Column("quarter", Integer),
    Column("is_weekend", Boolean)
)

dim_departments = Table(
    "dim_departments", metadata,
    Column("id", Integer, primary_key=True),
    Column("name", String(50)),
    Column("head_prof_user_id", Integer, nullable=True)
)

dim_courses = Table(
    "dim_courses", metadata,
    Column("id", Integer, primary_key=True),
    Column("code", String(20)),
    Column("name", String(50)),
    Column("description", TEXT),
    Column("credits", Integer),
    Column("department_id", Integer, ForeignKey("dim_departments.id"))
)

dim_course_offerings = Table(
    "dim_course_offerings", metadata,
    Column("id", Integer, primary_key=True),
    Column("course_id", Integer, ForeignKey("dim_courses.id")),
    Column("semester_id", Integer),
    Column("professor_user_id", Integer),
    Column("start_date", Date, ForeignKey("dim_date.date_key")),
    Column("end_date", Date, ForeignKey("dim_date.date_key")),
    Column("start_date_date_key", Integer, ForeignKey("dim_date.date_key")),
    Column("end_date_date_key", Integer, ForeignKey("dim_date.date_key"))
)

dim_students = Table(
    "dim_students", metadata,
    Column("user_id", Integer, primary_key=True),
    Column("full_name", String(100)),
    Column("birth_date", Date),
    Column("enroll_date_key", Integer, ForeignKey("dim_date.date_key")),
    Column("expected_graduation_date_key", Integer, ForeignKey("dim_date.date_key")),
    Column("academic_year", String(50)),
    Column("current_gpa", Numeric(3,2)),
    Column("passed_credits", Integer),
    Column("registered_credits", Integer),
    Column("department_id", Integer, ForeignKey("dim_departments.id")),
)

fact_attendance = Table(
    "fact_attendance", metadata,
    Column("student_user_id", Integer, ForeignKey("dim_students.user_id")),
    Column("course_offering_id", Integer, ForeignKey("dim_course_offerings.id")),
    Column("session_date", String(50)),
    Column("status", String(20)),
    Column("recorded_at", String(100)),
    Column("session_date_key", Integer, ForeignKey("dim_date.date_key")),
    Column("recorded_at_date_key", Integer, ForeignKey("dim_date.date_key"))
)

fact_students = Table(
    "fact_students", metadata,
    Column("user_id", Integer, ForeignKey("dim_students.user_id")),
    Column("current_gpa", Numeric(3,2)),
    Column("passed_credits", Integer),
    Column("enroll_date_date_key", Integer, ForeignKey("dim_date.date_key"))
)

fact_enrollments = Table(
    "fact_enrollments", metadata,
    Column("student_user_id", Integer, ForeignKey("dim_students.user_id")),
    Column("course_offering_id", Integer, ForeignKey("dim_course_offerings.id")),
    Column("enrolled_at_date_key", Integer, ForeignKey("dim_date.date_key"))
)

metadata.create_all(engine)

In [ ]:
df_dim_date.to_sql('dim_date', con=engine, if_exists='append', index=False)
df_dim_department.to_sql('dim_departments', con=engine, if_exists='append', index=False)
df_dim_courses.to_sql('dim_course', con=engine, if_exists='append', index=False)
df_dim_course_offerings.to_sql('dim_course_offerings', con=engine, if_exists='append', index=False)
df_dim_students.to_sql('dim_students', con=engine, if_exists='append', index=False)
df_fact_attendance.to_sql('fact_attendance', con=engine, if_exists='append', index=False)
df_fact_student.to_sql('fact_students', con=engine, if_exists='append', index=False)
df_fact_enrollment.to_sql('fact_enrollments', con=engine, if_exists='append', index=False)